# L03 · Mem0（2024）

## 30 秒核心

> Mem0 = **三段式 memory layer**：
> - **Extract**：LLM 抽事实 from conversation
> - **Update**：与 existing memory 比对，conflict → update / 新 → add
> - **Retrieve**：query → relevant memory back to LLM

## API

```python
from mem0 import Memory
m = Memory()

# Add (会 extract facts)
m.add("I prefer Anthropic Claude over GPT.", user_id="alice")

# Search
results = m.search("What does alice prefer?", user_id="alice")
# → [{"memory":"Alice prefers Anthropic Claude over GPT", "score":0.92}]
```

## 流程

```
Conversation:
  "I just switched from OpenAI to Anthropic Claude."
       ↓
Extract LLM:
  Facts: ["User uses Anthropic Claude", "User previously used OpenAI"]
       ↓
Update LLM (compare to existing):
  Existing: "User uses OpenAI"
  Decision: UPDATE (conflict)
  New: "User uses Anthropic Claude (switched from OpenAI)"
       ↓
Store in vector + KG
```

## Update 决策

| 情况 | Action |
|------|--------|
| 新事实 | ADD |
| 与已有相同 | NONE |
| 与已有冲突 | UPDATE (新覆盖) |
| 不相关 noise | NONE |

LLM-as-judge 决 action。

## Mem0 强项

| 强 | 解释 |
|----|------|
| LLM extract 干净事实 | 比 raw chat 高质 |
| Conflict resolution | 自动覆盖 |
| Per-user / per-session 隔离 | 多租户友好 |
| OSS + cloud | 双轨 |

## Mem0 vs Letta

| 维度 | Mem0 | Letta |
|------|------|-------|
| Memory 单位 | extracted facts | message + archive |
| 更新机制 | extract + decide | LLM tool call |
| 抽象层 | 高 (事实级) | 低 (消息级) |
| Best for | 长尾用户 profile | 长对话 |

## 实现 (`mem0_mock.py` 预告)

```python
class Mem0Mock:
    def __init__(self):
        self.facts: dict[str, list[dict]] = {}  # user_id → facts

    def add(self, content: str, user_id: str):
        new_facts = self._extract(content)
        existing = self.facts.setdefault(user_id, [])
        for f in new_facts:
            action = self._decide_action(f, existing)
            if action == "ADD": existing.append({"text":f, ...})
            elif action == "UPDATE": ...

    def search(self, query: str, user_id: str, k=3):
        # cosine over facts
        ...
```

## 数字（Mem0 paper / blog）

| 设置 | retrieval P@5 |
|------|--------------:|
| Raw chat history | 60% |
| Mem0 extracted facts | 85% |

→ extract 后干净事实比原 chat 检索准。

## 退出条件

- 能默写 extract / update / retrieve 三段
- 能讲 4 update action
- 与 Letta 区分

## 一句话

> Mem0 = LLM extract → decide (ADD/UPDATE/NONE) → store —— 把对话蒸馏成 clean facts，retrieval P@5 +25pp。


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
